# Lasso - Simulation Study

In [1]:
### IMPORTS
# Core data manipulation and numerical computing libraries.
import pandas as pd
import numpy as np

# Plotting utilities used throughout the exploratory and final-fit plots.
import matplotlib.pyplot as plt
from IPython.display import display
plt.style.use('default')

# Linear models used as the base estimators for polynomial and spline regression.
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso

# Model-assessment metric used for every method comparison.
from sklearn.metrics import mean_squared_error

# Resampling utilities for train/test splits, k-fold CV, LOOCV, and CV predictions.
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import LeaveOneOut
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import cross_val_score

# Pipeline helper lets preprocessing steps and regression models act as one estimator.
from sklearn.pipeline import make_pipeline

# Feature constructors used for nonlinear regression models.
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import SplineTransformer
from sklearn.preprocessing import StandardScaler


In [2]:
### GLOBAL VARS
# A single seed keeps every random split and simulated data set reproducible.
RANDOM_SEED = 123


In [ ]:
def make_sparse_lasso_data(
    n=100,
    p=10,
    n_active=5,
    beta_min=1.0,
    beta_max=5.0,
    noise=1.0,
    random_state=RANDOM_SEED
):
    # Generate sparse regression data for lasso.
    rng = np.random.default_rng(random_state)

    # Generate predictors.
    X = rng.normal(0.0, 1.0, size=(n, p))

    # Start with all coefficients equal to zero.
    beta = np.zeros(p)

    # Randomly choose which predictors are active.
    active_idx = rng.choice(p, size=n_active, replace=False)

    # Randomly choose coefficient magnitudes between beta_min and beta_max.
    magnitudes = rng.uniform(beta_min, beta_max, size=n_active)

    # Randomly assign positive or negative signs.
    signs = rng.choice([-1.0, 1.0], size=n_active)

    # Fill in the active coefficients.
    beta[active_idx] = magnitudes * signs

    # Generate response.
    y = X @ beta + rng.normal(0.0, noise, size=n)

    # Put data into a DataFrame.
    columns = [f"x{j + 1}" for j in range(p)]
    dat = pd.DataFrame(X, columns=columns)
    dat["y"] = y

    return X, y, beta, active_idx, dat

In [ ]:
# Many irrelevant predictors
X1, y1, beta1, active_idx1, data1 = make_sparse_lasso_data(p=20, n_active=2)

# Mostly relevant predictors
X2, y2, beta2, active_idx2, data2 = make_sparse_lasso_data(p=20, n_active=18)

# High noise
X3, y4, beta3, active_idx3, data3 = make_sparse_lasso_data(3)

# Low noise
X4, y4, beta4, active_idx4, data4 = make_sparse_lasso_data(noise=0.5)

,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,y
0,-0.989121,-0.367787,1.287925,0.193974,0.920231,0.577104,-0.636464,0.541952,-0.316595,-0.322389,-2.806888
1,0.097167,-1.525930,1.192166,-0.671090,1.000269,0.136321,1.532033,-0.659969,-0.311795,0.337769,-1.384216
2,-2.207471,0.827921,1.541630,1.126807,0.754770,-0.145978,1.281902,1.074031,0.392621,0.005114,-2.450546
3,-0.361767,-1.230232,1.226229,-2.172044,-0.370147,0.164380,0.859881,1.761661,0.993324,-0.291521,-2.476803
4,0.728128,-1.261600,1.429939,-0.156475,-0.673759,-0.639060,-0.061361,-0.392785,2.289910,-0.718181,-7.652946
...,...,...,...,...,...,...,...,...,...,...,...
95,1.967809,-1.672966,0.043084,-1.577245,0.074545,-0.306236,2.040681,0.543948,-1.060898,1.563224,6.249478
96,-1.540861,0.697686,1.818897,0.407416,0.731330,0.552721,-0.111986,0.312185,-1.473524,0.600501,2.843732
97,-0.659069,0.880500,-0.288378,-0.071044,-0.180547,-0.778047,0.564389,-0.225363,-1.011410,-0.428560,5.478312
98,-0.150977,0.427214,0.228927,-0.442497,0.301541,2.261331,-0.252522,-2.805756,-1.129308,-0.642893,-1.809357
